<a href="https://colab.research.google.com/github/SebastianoColombini/NN_DL_project/blob/main/prova.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Import section
from google.colab import drive
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [8]:
version = {
    "version_name": "alpha",
    "lr": None
}

In [9]:
# IMPORT DI GOOGLE DRIVE
drive.mount('/content/drive')

OUTPUT_DIR = "/content/drive/MyDrive/NN_DL_Proj"
VERSION_DIR = os.path.join(OUTPUT_DIR, version["version_name"])
SNAPSHOTS_DIR = os.path.join(VERSION_DIR, "snapshots")


os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(VERSION_DIR, exist_ok=True)
os.makedirs(SNAPSHOTS_DIR, exist_ok=True)

# Save the version JSON to a file in VERSION_DIR
version_file_path = os.path.join(VERSION_DIR, "version.json")
with open(version_file_path, 'w') as f:
    json.dump(version, f, indent=4)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# LOSS UTILS
def normalize(saliency_n)
def kld_loss()
def cc_loss()
def sim_loss()
def single_branch_loss()
def total_loss()

In [6]:
# ENCODER
class Encoder(nn.Module):
  def __init__(self):
    super().__init__()
    self.kernel_size=3
    self.stride=2
    self.padding=1

    self.encoder_cnn1 = nn.Sequential(
        nn.conv2d(3, 64, kernel_size=self.kernel_size, stride=self.stride, padding=self.padding),
        nn.ReLU(True)
    )
    self.encoder_cnn2 = nn.Sequential(
        nn.conv2d(64, 128, kernel_size=self.kernel_size, stride=self.stride, padding=self.padding),
        nn.ReLU(True)
    )
    self.encoder_cnn3 = nn.Sequential(
        nn.conv2d(128, 256, kernel_size=self.kernel_size, stride=self.stride, padding=self.padding),
        nn.ReLU(True)
    )
    self.encoder_cnn4 = nn.Sequential(
        nn.conv2d(256, 512, kernel_size=self.kernel_size, stride=self.stride, padding=self.padding),
        nn.ReLU(True)
    )
    self.encoder_cnn5 = nn.Sequential(
        nn.conv2d(512, 512, kernel_size=self.kernel_size, stride=self.stride, padding=self.padding),
        nn.ReLU(True)
    )

  def forward(self, x: torch.Tensor):
    out = []
    x = self.encoder_cnn1(x)
    x = self.encoder_cnn2(x)
    x = self.encoder_cnn3(x)
    out.append(x.copy())
    x = self.encoder_cnn4(x)
    out.append(x.copy())
    x = self.encoder_cnn5(x)
    out.append(x.copy())
    return out




In [ ]:
# DECODER 3
class Decoder3(nn.Module):
  def __init__(self, encoded_space_dim: int)
    super().__init__()
    self.decoder_cnn = nn.Sequential(
        nn.Conv2d(256, 1, kernel_size=1, stride=1, padding=0),
        nn.ReLU(True),
        nn.ConvTranspose2d(1, 1, kernel_size=16, stride=8, padding=4),
        nn.Sigmoid(True)
    )

  def forward(self, x: torch.Tensor):
    return self.decoder_cnn(x)

In [ ]:
# DECODER 4
class Decoder4(nn.Module):
  def __init__(self, encoded_space_dim: int)
    super().__init__()
    self.decoder_cnn = nn.Sequential(
        nn.Conv2d(512, 1, kernel_size=1, stride=1, padding=0),
        nn.ReLU(True),
        nn.ConvTranspose2d(1, 1, kernel_size=32, stride=16, padding=8),
        nn.Sigmoid(True)
    )

  def forward(self, x: torch.Tensor):
    return self.decoder_cnn(x)

In [ ]:
# DECODER 5
class Decoder5(nn.Module):
  def __init__(self, encoded_space_dim: int)
    super().__init__()
    self.decoder_cnn = nn.Sequential(
        nn.Conv2d(512, 1, kernel_size=1, stride=1, padding=0),
        nn.ReLU(True),
        nn.ConvTranspose2d(1, 1, kernel_size=32, stride=16, padding=8),
        nn.Sigmoid(True)
    )

  def forward(self, x: torch.Tensor):
    return self.decoder_cnn(x)

In [ ]:
# FINAL LAYER
class FinalCov(nn.Module):
  def __init__(self, encoded_space_dim: int)
    super().__init__()
    self.decoder_cnn = nn.Sequential(
        nn.ConvTranspose2d(3, 1, kernel_size=1, stride=1, padding=0),
        nn.Sigmoid(True)
    )

  def forward(self, x: torch.Tensor):
    return self.decoder_cnn(x)



In [ ]:
# MAIN PHASE OF TRAINING
# Istanziazione dei componenti e aggancio all'ottimizzatore
encoder = Encoder()
decoder3 = Decoder3()
decoder4 = Decoder4()
decoder5 = Decoder5()
final = FinalCov()


loss_fn = nn.MSELoss()
optimizer = optim.Adam(
    list(encoder.parameters()) +
    list(decoder3.parameters()) +
    list(decoder4.parameters()) +
    list(decoder5.parameters()) +
    list(final.parameters())
    , lr=0.001
)

# Rilevamento hardware (CUDA / CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoder.to(device)
decoder3.to(device)
decoder4.to(device)
decoder5.to(device)
final.to(device)

# Loop delle epoche
num_epochs = 20
for epoch in range(num_epochs):

    # Attivazione dei moduli in modalità addestramento
    encoder.train()
    decoder3.train()
    decoder4.train()
    decoder5.train()
    final.train()

    for image_batch, _ in dataloader: # Le label di classificazione vengono ignorate

        # Allocazione del batch sul dispositivo di calcolo
        image_batch = image_batch.to(device)

        # Passaggio in avanti (Forward Pass)
        latent_features = encoder(image_batch)
        saliency_3 = decoder3(latent_features[0])
        saliency_4 = decoder4(latent_features[1])
        saliency_5 = decoder5(latent_features[2])
        fused_input=torch.cat((saliency_3, saliency_4, saliency_5),dim =1)
        reconstructed_images = final(fused_input)

        loss = loss_fn(reconstructed_images, image_batch)

        optimizer.zero_grad()  # Svuotamento dei gradienti accumulati nei tensori
        loss.backward()       # Calcolo dei nuovi gradienti tramite backpropagation
        optimizer.step()       # Aggiornamento dei parametri della rete